# ArXiv Paper Expert — RAG-Powered AI Research Q&A

A Retrieval-Augmented Generation (RAG) system that lets you ask natural language questions about the latest AI/ML research papers from ArXiv.

## How It Works
```
ArXiv RSS Feed → Scrape Papers → Chunk Text → Embed → ChromaDB
                                                           ↓
User Question → Embed Question → Semantic Search → Top-5 Chunks → GPT-4o-mini → Answer
```

## Stack
- **Data**: ArXiv RSS feed (cs.AI + cs.LG categories)
- **Embeddings**: `sentence-transformers/all-MiniLM-L6-v2` (local, free)
- **Vector Database**: ChromaDB (persistent, local)
- **LLM**: GPT-4o-mini via OpenAI API

In [ ]:
# Install all required packages
import sys
!{sys.executable} -m pip install feedparser beautifulsoup4 chromadb sentence-transformers openai python-dotenv --quiet
print("✅ Packages installed!")

## Step 1: Imports & Setup

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import os
import re
import time
import requests
import feedparser
from bs4 import BeautifulSoup
from pathlib import Path
from sentence_transformers import SentenceTransformer
import chromadb
import openai
from dotenv import load_dotenv

print("✅ Imports complete!")

## Step 2: Paper Collection
Fetch the latest AI/ML papers from two ArXiv categories:
- `cs.AI` — Artificial Intelligence
- `cs.LG` — Machine Learning

Each paper's title, abstract, authors, and URL are saved as a `.txt` file.

In [ ]:
papers_dir = Path("./papers")
papers_dir.mkdir(exist_ok=True)

# ArXiv RSS feeds — cs.AI and cs.LG (Machine Learning)
RSS_FEEDS = {
    "cs.AI": "https://arxiv.org/rss/cs.AI",
    "cs.LG": "https://arxiv.org/rss/cs.LG",
}

def fetch_arxiv_papers(rss_url, category, max_papers=15):
    """Fetch and save papers from an ArXiv RSS feed"""
    print(f"\nFetching {category} papers from ArXiv RSS...")
    feed = feedparser.parse(rss_url)
    print(f"Found {len(feed.entries)} papers")

    saved = 0
    for i, entry in enumerate(feed.entries[:max_papers]):
        title    = entry.title.replace("\n", " ").strip()
        abstract = entry.summary.replace("\n", " ").strip()
        url      = entry.link
        authors  = entry.get("authors", [])
        author_names = ", ".join([a.get("name", "") for a in authors]) if authors else "N/A"
        date     = entry.get("published", "")[:10]

        # Clean up HTML tags from abstract if present
        abstract = BeautifulSoup(abstract, "html.parser").get_text()

        # Save as structured text file
        safe_title = re.sub(r'[^a-zA-Z0-9]+', '_', title)[:60]
        filename   = f"{category}_{i+1:03d}_{safe_title}.txt"
        filepath   = papers_dir / filename

        with open(filepath, "w", encoding="utf-8") as f:
            f.write(f"Title: {title}\n")
            f.write(f"Authors: {author_names}\n")
            f.write(f"Date: {date}\n")
            f.write(f"Category: {category}\n")
            f.write(f"URL: {url}\n")
            f.write("=" * 60 + "\n\n")
            f.write(f"ABSTRACT:\n{abstract}\n")

        print(f"  ✅ [{i+1:02d}] {title[:70]}...")
        saved += 1
        time.sleep(0.2)

    return saved

# Fetch from both categories
total = 0
for category, rss_url in RSS_FEEDS.items():
    total += fetch_arxiv_papers(rss_url, category, max_papers=15)

print(f"\n✅ Total papers saved: {total} → ./papers/")

## Step 3: Chunking
Split each paper into overlapping chunks.
Since ArXiv papers have shorter abstracts than blog posts, we use smaller chunks (300 words) for more precise retrieval.

In [ ]:
def chunk_text(text, chunk_size=300, overlap=30):
    """Split text into overlapping chunks of ~chunk_size words"""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

# Load all paper files and chunk them
all_chunks   = []
all_metadata = []

files = sorted(papers_dir.glob("*.txt"))
print(f"Loading {len(files)} paper files...\n")

for filepath in files:
    with open(filepath, "r", encoding="utf-8") as f:
        raw = f.read()

    # Parse header metadata
    lines    = raw.split("\n")
    title    = lines[0].replace("Title: ", "").strip()
    authors  = lines[1].replace("Authors: ", "").strip()
    date     = lines[2].replace("Date: ", "").strip()
    category = lines[3].replace("Category: ", "").strip()
    url      = lines[4].replace("URL: ", "").strip()

    # Chunk the content below the header
    content = "\n".join(lines[6:])
    chunks  = chunk_text(content)

    for i, chunk in enumerate(chunks):
        all_chunks.append(chunk)
        all_metadata.append({
            "title":    title,
            "authors":  authors[:200],  # ChromaDB metadata has length limit
            "date":     date,
            "category": category,
            "url":      url,
            "source":   filepath.name,
            "chunk":    i
        })

print(f"Total papers:  {len(files)}")
print(f"Total chunks:  {len(all_chunks)}")
print(f"Avg chunk size: {sum(len(c.split()) for c in all_chunks) // max(len(all_chunks),1)} words")
print(f"\nSample chunk:")
print("-" * 50)
print(all_chunks[0][:300] + "...")

## Step 4: Vector Database Setup
Embed every chunk using `sentence-transformers` and store in ChromaDB for fast semantic search.

In [ ]:
# Load embedding model — free, local, no API key needed
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model ready (384-dimensional vectors)\n")

# Set up ChromaDB persistent vector database
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Fresh start — delete existing collection if re-running
try:
    chroma_client.delete_collection("arxiv_papers")
    print("Cleared existing collection.")
except:
    pass

collection = chroma_client.create_collection(
    name="arxiv_papers",
    metadata={"hnsw:space": "cosine"}  # cosine similarity for semantic search
)

# Embed and store in batches
print(f"Embedding {len(all_chunks)} chunks into ChromaDB...")
BATCH_SIZE = 50

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch_chunks   = all_chunks[i:i + BATCH_SIZE]
    batch_metadata = all_metadata[i:i + BATCH_SIZE]
    batch_ids      = [f"chunk_{j}" for j in range(i, i + len(batch_chunks))]
    embeddings     = embedder.encode(batch_chunks).tolist()

    collection.add(
        documents=batch_chunks,
        embeddings=embeddings,
        metadatas=batch_metadata,
        ids=batch_ids
    )
    print(f"  Stored chunks {i} → {i + len(batch_chunks)}")

print(f"\n✅ Vector database ready — {collection.count()} chunks stored in ChromaDB")

## Step 5: RAG Query Pipeline
Full pipeline:
1. Embed the user question into a vector
2. Search ChromaDB for top-5 semantically similar paper chunks
3. Send question + retrieved chunks to Claude as context
4. Claude generates a grounded answer with paper citations

In [ ]:
# Load OpenAI API key from .env file — no manual paste needed
load_dotenv()  # looks for .env in current directory (arxiv-expert-rag/)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Create a .env file with your key (see .env.example).")

client = openai.OpenAI(api_key=OPENAI_API_KEY)
print(f"✅ OpenAI API key loaded: {OPENAI_API_KEY[:8]}...{OPENAI_API_KEY[-4:]}")

def retrieve(question, top_k=5):
    """Semantic search — find the most relevant paper chunks for a question"""
    question_embedding = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

def ask(question):
    """
    Full RAG pipeline:
    Question → Retrieve relevant chunks → Generate grounded answer
    """
    print(f"{'='*60}")
    print(f"❓ Question: {question}")
    print(f"{'='*60}\n")

    # Step 1 — Retrieve
    chunks, metadatas = retrieve(question)

    # Step 2 — Build context with source labels
    context_parts = []
    for i, (chunk, meta) in enumerate(zip(chunks, metadatas)):
        context_parts.append(
            f"[Paper {i+1}: {meta['title']}]\n"
            f"Authors: {meta['authors']}\n"
            f"Date: {meta['date']} | Category: {meta['category']}\n\n"
            f"{chunk}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # Step 3 — Generate answer with GPT-4o-mini
    prompt = f"""You are an expert AI research assistant with deep knowledge of machine learning and artificial intelligence.

Your job is to answer questions about recent ArXiv research papers using ONLY the provided paper excerpts below.

Rules:
- Answer based strictly on the provided excerpts
- Cite the paper title when referencing specific findings
- If the answer is not in the excerpts, say "This topic wasn't covered in the retrieved papers."
- Be concise and technical — the audience is engineers and researchers

PAPER EXCERPTS:
{context}

QUESTION: {question}

ANSWER:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}]
    )

    answer = response.choices[0].message.content
    print(f"🤖 Answer:\n{answer}\n")

    # Show sources
    print("📚 Papers used as sources:")
    seen = set()
    for meta in metadatas:
        if meta['title'] not in seen:
            print(f"  [{meta['category']}] {meta['title']}")
            print(f"           {meta['url']}")
            seen.add(meta['title'])

    return answer

print("✅ RAG pipeline ready!")

## Step 6: Ask Research Questions
Query the system with natural language questions about recent AI/ML research.

In [ ]:
ask("What are the latest techniques for improving LLM efficiency?")

In [ ]:
ask("What recent research has been done on fine-tuning and transfer learning?")

In [ ]:
ask("What are researchers working on in the area of AI agents and reasoning?")

In [ ]:
# Try your own research question here
ask("What evaluation benchmarks are being used for large language models?")